In [1]:
from google.colab import userdata

# HOW TO SET THIS UP (one-time):
# 1. Look at the left sidebar in Colab
# 2. Click the 🔑 "Secrets" icon
# 3. Click "Add new secret"
# 4. Name it exactly:  GEMINI_KEY
# 5. Paste your API key as the value
# 6. Toggle "Notebook access" ON

MY_GEMINI_KEY = userdata.get('GEMINI_KEY')

if MY_GEMINI_KEY:
    print("API Key loaded securely.")
else:
    print("ERROR: Key not found. Check your Secrets tab.")


API Key loaded securely.


In [2]:
import pandas as pd
import io
from google.colab import files

CHORD_DICT = {
    '1': 'I',  '2': 'ii', '3': 'iii', '4': 'IV',
    '5': 'V',  '6': 'vi', '7': 'vii°'
}

def translate_cp(cp_string):
    try:
        return ' - '.join(CHORD_DICT.get(c.strip(), c.strip()) for c in str(cp_string).split(','))
    except:
        return str(cp_string)

print("Upload your 3 CSV files again:")
uploaded = files.upload()

all_dfs = []
for filename, content in uploaded.items():
    df = pd.read_csv(io.BytesIO(content))

    # Keep mode column this time (0=minor, 1=major)
    required = {'cp', 'valence', 'energy', 'artist', 'song'}
    if required.issubset(df.columns):
        cols = list(required)
        if 'mode' in df.columns:
            cols.append('mode')
        clean = df[cols].dropna()
        all_dfs.append(clean)
        print(f"  Loaded {filename}: {len(clean)} rows")
    elif 'cp' in df.columns:
        clean = df[['cp']].dropna()
        all_dfs.append(clean)
        print(f"  Loaded {filename} (cp only): {len(clean)} rows")
    else:
        print(f"  WARNING: {filename} skipped.")

music_db = pd.concat(all_dfs, ignore_index=True)

if 'artist' in music_db.columns and 'song' in music_db.columns:
    music_db = music_db.drop_duplicates(subset=['artist', 'song'])

music_db['roman'] = music_db['cp'].apply(translate_cp)

print(f"\nMuseTune Database ready: {len(music_db)} unique songs.")
print(f"Columns: {music_db.columns.tolist()}")
print(music_db.head())


Upload your 3 CSV files again:


Saving three_four_five_pruned.csv to three_four_five_pruned.csv
Saving four_chord_songs.csv to four_chord_songs.csv
Saving five_chord_songs.csv to five_chord_songs.csv
Saving three_four_five.csv to three_four_five.csv
  Loaded three_four_five_pruned.csv: 12511 rows
  Loaded four_chord_songs.csv: 11931 rows
  Loaded five_chord_songs.csv: 9884 rows
  Loaded three_four_five.csv: 36300 rows

MuseTune Database ready: 3263 unique songs.
Columns: ['energy', 'song', 'valence', 'artist', 'cp', 'mode', 'roman']
   energy                      song  valence        artist     cp  mode  \
0   0.856                    cryin'    0.486     aerosmith  1,5,6   1.0   
1   0.657                     flake    0.964  jack johnson  1,5,6   1.0   
2   0.826                  firework    0.649    katy perry  1,5,6   1.0   
3   0.715  i want to hold your hand    0.866   the beatles  1,5,6   1.0   
4   0.403                 let it be    0.410   the beatles  1,5,6   1.0   

        roman  
0  I - V - vi  
1  I - V -

In [3]:
EMOTION_FILTERS = {
    "sadness":        {"valence_max": 0.30, "energy_max": 0.45},
    "grief":          {"valence_max": 0.25, "energy_max": 0.35},
    "joy":            {"valence_min": 0.70, "energy_min": 0.60},
    "excitement":     {"valence_min": 0.65, "energy_min": 0.75},
    "anger":          {"valence_max": 0.40, "energy_min": 0.72, "mode": 0},  # minor key
    "fear":           {"valence_max": 0.38, "energy_min": 0.50, "energy_max": 0.72, "mode": 0},
    "calmness":       {"valence_min": 0.45, "energy_max": 0.40},
    "love":           {"valence_min": 0.55, "energy_min": 0.30, "energy_max": 0.60},
    "disappointment": {"valence_max": 0.40, "energy_max": 0.50},
}

GOEMOTIONS_TO_MUSICAL = {
    "joy": "joy",               "amusement": "joy",          "optimism": "joy",
    "excitement": "excitement", "pride": "excitement",
    "sadness": "sadness",       "disappointment": "disappointment",
    "grief": "grief",           "remorse": "sadness",        "embarrassment": "sadness",
    "anger": "anger",           "annoyance": "anger",        "disgust": "anger",
    "fear": "fear",             "nervousness": "fear",
    "relief": "calmness",       "approval": "calmness",      "gratitude": "calmness",
    "calmness": "calmness",     # ← fix: direct mapping added
    "love": "love",             "caring": "love",            "admiration": "love",
    "neutral": "calmness",      "surprise": "joy",           "curiosity": "joy",
    "realization": "calmness",  "confusion": "fear",
}

def query_db_for_emotion(detected_emotion, db, top_n=3):
    musical_emotion = GOEMOTIONS_TO_MUSICAL.get(detected_emotion, "joy")
    rules = EMOTION_FILTERS.get(musical_emotion, {})

    filtered = db.copy()

    if 'valence' in filtered.columns and 'energy' in filtered.columns:
        if "valence_min" in rules:
            filtered = filtered[filtered['valence'] >= rules['valence_min']]
        if "valence_max" in rules:
            filtered = filtered[filtered['valence'] <= rules['valence_max']]
        if "energy_min" in rules:
            filtered = filtered[filtered['energy'] >= rules['energy_min']]
        if "energy_max" in rules:
            filtered = filtered[filtered['energy'] <= rules['energy_max']]

    # Filter by mode (minor=0 / major=1) only if column exists
    if "mode" in rules and 'mode' in filtered.columns:
        filtered = filtered[filtered['mode'] == rules['mode']]

    # Fallback if too few results
    if len(filtered) < 5:
        print(f"  Note: strict filter returned {len(filtered)} songs, applying relaxed filter.")
        filtered = db.copy()
        if 'valence' in db.columns:
            v_mid = (rules.get('valence_min', 0.5) + rules.get('valence_max', 0.5)) / 2
            filtered = filtered[(filtered['valence'] - v_mid).abs() < 0.25]

    top_progressions = filtered['cp'].value_counts().head(top_n)

    results = []
    for cp, count in top_progressions.items():
        examples = (
            filtered[filtered['cp'] == cp][['artist', 'song', 'valence', 'energy']]
            .drop_duplicates()
            .head(2)
        )
        results.append({
            "cp": cp,
            "roman": translate_cp(cp),
            "count": count,
            "examples": examples
        })

    return musical_emotion, results


def print_emotion_query(detected_emotion, db):
    musical_emotion, results = query_db_for_emotion(detected_emotion, db)
    print(f"\n--- [{detected_emotion.upper()} → bucket: {musical_emotion.upper()}] ---")
    for r in results:
        print(f"  {r['roman']} ({r['count']} songs)")
        for _, row in r['examples'].iterrows():
            print(f"    ↳ {row['artist'].title()} — '{row['song'].title()}' (Val:{row['valence']:.2f}, Nrg:{row['energy']:.2f})")
    best = results[0]['roman'] if results else "I - V - vi - IV"
    print(f"\n  Best match: {best}")
    return best

# Test all emotions
for emotion in ["sadness", "joy", "anger", "fear", "calmness"]:
    print_emotion_query(emotion, music_db)



--- [SADNESS → bucket: SADNESS] ---
  IV - V - I (11 songs)
    ↳ Kenny Chesney — 'The Road And The Radio' (Val:0.19, Nrg:0.43)
    ↳ John Lennon — 'Imagine' (Val:0.17, Nrg:0.26)
  IV - I - V (9 songs)
    ↳ Counting Crows — 'Walkaways' (Val:0.17, Nrg:0.29)
    ↳ Johnny Cash — 'Hurt' (Val:0.17, Nrg:0.40)
  IV - V - vi (9 songs)
    ↳ Elvis Presley — 'Can'T Help Falling In Love' (Val:0.15, Nrg:0.24)
    ↳ Elvis Presley — 'I Can'T Help Falling In Love' (Val:0.10, Nrg:0.21)

  Best match: IV - V - I

--- [JOY → bucket: JOY] ---
  I - V - vi - IV - I (33 songs)
    ↳ Flo Rida — 'Whistle' (Val:0.74, Nrg:0.94)
    ↳ Matchbox 20 — 'Real World' (Val:0.88, Nrg:0.76)
  I - V - IV - I - V (30 songs)
    ↳ Bruno Mars — 'The Lazy Song' (Val:0.95, Nrg:0.80)
    ↳ Creedence Clearwater Revival — 'Bad Moon Rising' (Val:0.94, Nrg:0.77)
  I - IV - V - I - IV (19 songs)
    ↳ One Direction — 'What Makes You Beautiful' (Val:0.87, Nrg:0.77)
    ↳ Train — 'Save Me San Francisco' (Val:0.89, Nrg:0.94)

  Best

In [4]:
# Step 1: Clone the repo and explore its structure
!git clone https://github.com/smashub/choco.git

import os

partitions_path = "choco/partitions"

if os.path.exists(partitions_path):
    print("Available partition sources:")
    for folder in sorted(os.listdir(partitions_path)):
        folder_path = os.path.join(partitions_path, folder)
        if os.path.isdir(folder_path):
            file_count = sum(len(files) for _, _, files in os.walk(folder_path))
            print(f"  {folder} — {file_count} files")
else:
    print("Partitions folder not found.")


Cloning into 'choco'...
remote: Enumerating objects: 494683, done.
remote: Counting objects: 100% (234068/234068), done.
remote: Compressing objects: 100% (4602/4602), done.
remote: Total 494683 (delta 233488), reused 229909 (delta 229454), pack-reused 260615 (from 1)
Receiving objects: 100% (494683/494683), 1.09 GiB | 22.54 MiB/s, done.
Resolving deltas: 100% (461049/461049), done.
Updating files: 100% (157467/157467), done.
Available partition sources:
  biab-internet-corpus — 9995 files
  billboard — 2685 files
  chordify — 114 files
  ireal-pro — 80167 files
  isophonics — 538 files
  jaah — 215 files
  jazz-corpus — 170 files
  mozart-piano-sonatas — 179 files
  nottingham — 2034 files
  real-book — 8554 files
  robbie-williams — 267 files
  rock-corpus — 2220 files
  rwc-pop — 318 files
  schubert-winterreise — 1947 files
  uspop2002 — 599 files
  weimar — 1835 files
  when-in-rome — 6424 files
  wikifonia — 18914 files


In [5]:
!pip install -q jams  # ← ADD THIS LINE

import jams
import os
import pandas as pd

# ── Harte converter (billboard, ireal-forum) ──────────────────────────
NOTE_TO_SEMI = {
    'C':0, 'C#':1, 'Db':1, 'D':2, 'D#':3, 'Eb':3,
    'E':4, 'Fb':4, 'F':5, 'F#':6, 'Gb':6, 'G':7,
    'G#':8, 'Ab':8, 'A':9, 'A#':10, 'Bb':10, 'B':11, 'Cb':11
}
# ... rest of your code stays exactly the same

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 5.8 MB/s eta 0:00:00


In [6]:
import jams
import os
import pandas as pd

# ── Harte converter (billboard, ireal-forum) ──────────────────────────
NOTE_TO_SEMI = {
    'C':0, 'C#':1, 'Db':1, 'D':2, 'D#':3, 'Eb':3,
    'E':4, 'Fb':4, 'F':5, 'F#':6, 'Gb':6, 'G':7,
    'G#':8, 'Ab':8, 'A':9, 'A#':10, 'Bb':10, 'B':11, 'Cb':11
}
INTERVAL_TO_DEGREE = {0:1, 2:2, 4:3, 5:4, 7:5, 9:6, 11:7}
DEGREE_TO_ROMAN    = {1:'I', 2:'II', 3:'III', 4:'IV', 5:'V', 6:'VI', 7:'VII'}

def harte_to_roman(chord_str, key_root):
    try:
        root, quality = (chord_str.split(':', 1) + ['maj'])[:2]
        chord_semi = NOTE_TO_SEMI.get(root)
        key_semi   = NOTE_TO_SEMI.get(key_root, 0)
        if chord_semi is None:
            return None
        interval = (chord_semi - key_semi) % 12
        degree   = INTERVAL_TO_DEGREE.get(interval)
        if degree is None:
            return None
        roman = DEGREE_TO_ROMAN[degree]
        if any(quality.startswith(q) for q in ['min', 'dim', 'hdim']):
            roman = roman.lower()
        if quality == 'maj7':
            roman += 'maj7'
        elif quality.startswith('dim'):
            roman += '°'
        elif '7' in quality:
            roman += '7'
        return roman
    except:
        return None


# ── chord_roman converter (rock-corpus) ──────────────────────────────
def roman_label_to_clean(chord_str):
    """
    Extracts Roman numeral from 'G:I' or 'G:IV' format.
    Just takes everything after the colon.
    """
    try:
        if ':' in chord_str:
            return chord_str.split(':', 1)[1].strip()
        return chord_str.strip()
    except:
        return None


# ── iReal Pro converter (ireal-playlists) ────────────────────────────
def ireal_to_roman(chord_str, key_root):
    """
    Converts iReal Pro chord (e.g. 'F-7', 'Bb7', 'Eb^7') to Roman numeral.
    """
    try:
        chord_str = chord_str.strip()
        if not chord_str or chord_str in ('n', 'N', 'x', 'X'):
            return None

        # Extract root (1 or 2 characters)
        root = chord_str[0]
        rest = chord_str[1:]
        if rest and rest[0] in ('#', 'b'):
            root += rest[0]
            rest = rest[1:]

        chord_semi = NOTE_TO_SEMI.get(root)
        key_semi   = NOTE_TO_SEMI.get(key_root, 0)
        if chord_semi is None:
            return None

        interval = (chord_semi - key_semi) % 12
        degree   = INTERVAL_TO_DEGREE.get(interval)
        if degree is None:
            return None

        roman = DEGREE_TO_ROMAN[degree]

        # iReal quality markers
        is_minor = rest.startswith('-') or rest.startswith('m')
        if is_minor:
            roman = roman.lower()

        if '^7' in rest or '^' in rest:
            roman += 'maj7'
        elif rest.startswith('o') or rest.startswith('°'):
            roman = roman.lower() + '°'
        elif '7' in rest:
            roman += '7'

        return roman
    except:
        return None


# ── Universal JAMS folder parser ──────────────────────────────────────
def parse_jams_folder(jams_folder, source_name, max_files=None):
    if not os.path.exists(jams_folder):
        print(f"  Folder not found: {jams_folder}")
        return []

    jams_files = [f for f in os.listdir(jams_folder) if f.endswith('.jams')]
    if max_files:
        jams_files = jams_files[:max_files]

    results = []
    errors  = 0

    for filename in jams_files:
        try:
            jam    = jams.load(os.path.join(jams_folder, filename), validate=False)
            title  = str(jam.file_metadata.title  or '').lower().strip()
            artist = str(jam.file_metadata.artist or '').lower().strip()

            # Get key
            song_key = 'C'
            for ann in jam.annotations:
                if ann.namespace == 'key_mode' and len(ann.data) > 0:
                    song_key = str(ann.data[0].value).split(':')[0].strip()
                    break

            # Get chord sequence — handle all 3 namespace types
            chord_seq  = []
            chord_type = None
            for ann in jam.annotations:
                if ann.namespace in ('chord', 'chord_roman', 'chord_ireal'):
                    chord_type = ann.namespace
                    for obs in ann.data:
                        val = str(obs.value).strip()
                        if val not in ('N', 'X', 'n', 'x', '', 'nan'):
                            chord_seq.append(val)
                    break

            if len(chord_seq) < 3:
                continue

            # Convert to Roman numerals based on chord type
            if chord_type == 'chord_roman':
                roman_list = [roman_label_to_clean(c) for c in chord_seq]
            elif chord_type == 'chord_ireal':
                roman_list = [ireal_to_roman(c, song_key) for c in chord_seq]
            else:
                roman_list = [harte_to_roman(c, song_key) for c in chord_seq]

            roman_list = [r for r in roman_list if r]

            if len(roman_list) < 3:
                continue

            # Compress repeated chords
            compressed = [roman_list[0]]
            for chord in roman_list[1:]:
                if chord != compressed[-1]:
                    compressed.append(chord)

            results.append({
                'artist': artist,
                'song':   title,
                'cp':     ' - '.join(compressed[:8]),
                'source': source_name
            })

        except Exception:
            errors += 1

    print(f"  {source_name}: {len(results)} progressions ({errors} skipped)")
    return results


# ── Run all partitions ────────────────────────────────────────────────
print("Parsing ChoCo partitions...\n")

all_results  = []
all_results += parse_jams_folder("choco/partitions/billboard/choco/jams",              "billboard")
all_results += parse_jams_folder("choco/partitions/rock-corpus/choco/jams",            "rock-corpus")
all_results += parse_jams_folder("choco/partitions/ireal-pro/choco/forum/jams",        "ireal-forum",     max_files=2000)
all_results += parse_jams_folder("choco/partitions/ireal-pro/choco/playlists/jams",    "ireal-playlists", max_files=2000)

choco_db = pd.DataFrame(all_results).drop_duplicates(subset=['artist', 'song'])

print(f"\nChoCo Database ready: {len(choco_db)} unique songs")
print(f"\nSources breakdown:")
print(choco_db['source'].value_counts().to_string())

print("\nSample progressions:")
print(choco_db[['artist', 'song', 'cp', 'source']].head(10).to_string(index=False))


Parsing ChoCo partitions...

  billboard: 890 progressions (0 skipped)
  rock-corpus: 194 progressions (0 skipped)
  ireal-forum: 1153 progressions (0 skipped)
  ireal-playlists: 1996 progressions (0 skipped)

ChoCo Database ready: 3955 unique songs

Sources breakdown:
source
ireal-playlists    1910
ireal-forum        1142
billboard           729
rock-corpus         174

Sample progressions:
artist                                                     song                                         cp    source
                                               baby i'm burnin'          I - IV - I - IV - I - IV - I - IV billboard
                                                      that girl         i - V7 - i - iv - i7 - i - iv - i7 billboard
                                                     last child    I7 - IV7 - I7 - IV - I7 - IV7 - I7 - IV billboard
                                           just the way you are        I - ii7 - I - IV - I - ii7 - I - IV billboard
       have you seen

In [7]:
!pip install -q transformers torch

from transformers import pipeline

print("Loading RoBERTa emotion classifier...")
print("This may take a minute on first run.\n")

emotion_classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=3  # return top 3 emotions, not just top 1
)

def detect_emotion(text):
    """
    Runs text through RoBERTa and returns the dominant emotion.
    Also shows top 3 so we can see confidence levels.
    """
    print(f"Input: '{text}'")
    print()

    results = emotion_classifier(text)

    print("Top 3 detected emotions:")
    for r in results[0]:
        bar = '█' * int(r['score'] * 20)
        print(f"  {r['label']:<20} {r['score']:.2f}  {bar}")

    top_emotion = results[0][0]['label']
    top_score   = results[0][0]['score']

    print(f"\nDominant emotion: [{top_emotion.upper()}] (confidence: {top_score:.2f})")
    return top_emotion


# ── Test with 4 different journal entries ────────────────────────────
print("=" * 55)
print("TEST 1")
print("=" * 55)
detect_emotion("I just got the promotion I've been working toward for three years!")

print("\n" + "=" * 55)
print("TEST 2")
print("=" * 55)
detect_emotion("I've been sitting in my car in the rain thinking about how much I miss my old friends.")

print("\n" + "=" * 55)
print("TEST 3")
print("=" * 55)
detect_emotion("I can't believe they lied to me. I'm absolutely furious right now.")

print("\n" + "=" * 55)
print("TEST 4")
print("=" * 55)
detect_emotion("Everything feels quiet today. I'm just sitting by the window watching the rain.")


Loading RoBERTa emotion classifier...
This may take a minute on first run.



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

TEST 1
Input: 'I just got the promotion I've been working toward for three years!'

Top 3 detected emotions:
  excitement           0.57  ███████████
  joy                  0.44  ████████
  neutral              0.04  

Dominant emotion: [EXCITEMENT] (confidence: 0.57)

TEST 2
Input: 'I've been sitting in my car in the rain thinking about how much I miss my old friends.'

Top 3 detected emotions:
  sadness              0.87  █████████████████
  disappointment       0.13  ██
  grief                0.03  

Dominant emotion: [SADNESS] (confidence: 0.87)

TEST 3
Input: 'I can't believe they lied to me. I'm absolutely furious right now.'

Top 3 detected emotions:
  anger                0.80  ███████████████
  annoyance            0.17  ███
  neutral              0.10  █

Dominant emotion: [ANGER] (confidence: 0.80)

TEST 4
Input: 'Everything feels quiet today. I'm just sitting by the window watching the rain.'

Top 3 detected emotions:
  neutral              0.51  ██████████
  joy           

'neutral'

In [8]:
from google import genai
from google.genai import types
from google.colab import userdata

MY_GEMINI_KEY = userdata.get('GEMINI_KEY')

def build_prompt(emotion, chords):
    return f"""You are a multi-platinum songwriter and music theorist.
Write exactly ONE 4-line verse matching this emotional and harmonic profile.

Target Emotion: [{emotion.upper()}]
Chord Progression: [{chords}]

Rules:
1. Exactly 4 lines. No labels, no headers.
2. ABAB or AABB rhyme scheme. No forced or cliché rhymes.
3. Word rhythm must match the harmonic tension of the chords.
4. Output ONLY the 4 lines. Nothing else.

Write the verse:"""


def generate_lyrics(emotion, chords, api_key):
    """Sends emotion + chords to Gemini and returns 4-line verse."""
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Write the complete 4-line verse now.',
        config=types.GenerateContentConfig(
            system_instruction=build_prompt(emotion, chords),
            temperature=0.75,
        )
    )
    return response.text.strip()


def run_musetune(journal_entry, api_key, db):
    """
    Full MuseTune pipeline:
    Text → Emotion → Database Query → Chords → Lyrics
    """
    print("=" * 55)
    print("MUSETUNE PIPELINE")
    print("=" * 55)

    # Step 1: Detect emotion
    print(f"\n[1] EMOTION DETECTION")
    print(f"    Input: '{journal_entry[:80]}'" if len(journal_entry) > 80 else f"    Input: '{journal_entry}'")
    results      = emotion_classifier(journal_entry)
    top_emotion  = results[0][0]['label']
    top_score    = results[0][0]['score']
    print(f"    Detected: {top_emotion.upper()} (confidence: {top_score:.2f})")

    # Step 2: Query database for best chords
    print(f"\n[2] DATABASE QUERY")
    best_chords = print_emotion_query(top_emotion, db)

    # Step 3: Generate lyrics
    print(f"\n[3] LYRIC GENERATION")
    print(f"    Sending to Gemini...")
    lyrics = generate_lyrics(top_emotion, best_chords, api_key)

    print(f"\n{'='*55}")
    print(f"FINAL OUTPUT")
    print(f"{'='*55}")
    print(f"Emotion  : {top_emotion.upper()}")
    print(f"Chords   : {best_chords}")
    print(f"Lyrics   :")
    print(f"\n{lyrics}\n")
    print("=" * 55)

    return {
        "input":      journal_entry,
        "emotion":    top_emotion,
        "confidence": top_score,
        "chords":     best_chords,
        "lyrics":     lyrics
    }


# ── Test the full pipeline with 3 different journal entries ──────────

result1 = run_musetune(
    "I've been sitting in my car in the rain thinking about how much I miss my old friends.",
    MY_GEMINI_KEY,
    music_db
)

print("\n\n")

result2 = run_musetune(
    "I can't believe they lied to me. I'm absolutely furious right now.",
    MY_GEMINI_KEY,
    music_db
)

print("\n\n")

result3 = run_musetune(
    "I just got the promotion I've been working toward for three years!",
    MY_GEMINI_KEY,
    music_db
)


MUSETUNE PIPELINE

[1] EMOTION DETECTION
    Input: 'I've been sitting in my car in the rain thinking about how much I miss my old fr'
    Detected: SADNESS (confidence: 0.87)

[2] DATABASE QUERY

--- [SADNESS → bucket: SADNESS] ---
  IV - V - I (11 songs)
    ↳ Kenny Chesney — 'The Road And The Radio' (Val:0.19, Nrg:0.43)
    ↳ John Lennon — 'Imagine' (Val:0.17, Nrg:0.26)
  IV - I - V (9 songs)
    ↳ Counting Crows — 'Walkaways' (Val:0.17, Nrg:0.29)
    ↳ Johnny Cash — 'Hurt' (Val:0.17, Nrg:0.40)
  IV - V - vi (9 songs)
    ↳ Elvis Presley — 'Can'T Help Falling In Love' (Val:0.15, Nrg:0.24)
    ↳ Elvis Presley — 'I Can'T Help Falling In Love' (Val:0.10, Nrg:0.21)

  Best match: IV - V - I

[3] LYRIC GENERATION
    Sending to Gemini...

FINAL OUTPUT
Emotion  : SADNESS
Chords   : IV - V - I
Lyrics   :

The light just barely touches on the floor,
And leaves me lonely, wanting something more.
I wonder why the silence feels so deep,
While all the promises are left to sleep.




MUSETUNE PI

In [9]:
!pip install -q music21

from music21 import roman, key

def score_harmonic_validity(chord_progression_str, key_str='C'):
    """
    Scores how musically valid a chord progression is.
    Returns a score from 0.0 to 1.0

    Checks:
    1. Parse rate    — can Music21 understand all the chords?
    2. Resolution    — does the progression contain V→I or vii°→I movements?
    3. Tonic start   — does it start or end on I (home base)?
    """
    k      = key.Key(key_str)
    chords = [c.strip() for c in chord_progression_str.split(' - ')]

    # ── Check 1: Parse rate ──────────────────────────────────────────
    parsed       = []
    valid_parses = 0
    for numeral in chords:
        try:
            rn = roman.RomanNumeral(numeral, k)
            parsed.append(rn)
            valid_parses += 1
        except Exception:
            parsed.append(None)

    parse_score = valid_parses / len(chords) if chords else 0

    # ── Check 2: Resolution (V→I or vii°→I) ─────────────────────────
    resolution_count = 0
    total_transitions = len(parsed) - 1

    if total_transitions > 0:
        for i in range(len(parsed) - 1):
            curr = parsed[i]
            nxt  = parsed[i + 1]
            if curr and nxt:
                curr_fig = str(curr.figure).upper().replace('°', '')
                nxt_fig  = str(nxt.figure).upper()
                if curr_fig in ('V', 'V7', 'VII') and nxt_fig in ('I', 'IMAJ7'):
                    resolution_count += 1

    resolution_score = resolution_count / total_transitions if total_transitions > 0 else 0

    # ── Check 3: Tonic anchor (starts or ends on I) ──────────────────
    tonic_score = 0
    if parsed:
        first = parsed[0]
        last  = parsed[-1]
        if first and str(first.figure).upper() in ('I', 'IMAJ7'):
            tonic_score += 0.5
        if last and str(last.figure).upper() in ('I', 'IMAJ7'):
            tonic_score += 0.5

    # ── Final weighted score ─────────────────────────────────────────
    final_score = (parse_score * 0.5) + (resolution_score * 0.3) + (tonic_score * 0.2)

    return {
        "score":          round(final_score, 3),
        "parse_rate":     round(parse_score, 3),
        "resolution":     round(resolution_score, 3),
        "tonic_anchor":   round(tonic_score, 3),
        "valid_chords":   valid_parses,
        "total_chords":   len(chords)
    }


# ── Test on real progressions ────────────────────────────────────────
test_cases = {
    "Sadness  (IV - V - I)":          "IV - V - I",
    "Anger    (I - V - vi - IV - I)": "I - V - vi - IV - I",
    "Excitement (I - V - vi - IV - I)":"I - V - vi - IV - I",
    "Jazz     (Imaj7 - ii7 - V7 - I)":"Imaj7 - ii7 - V7 - I",
    "Minor    (i - iv - i)":          "i - iv - i",
    "Invalid  (X - ZZ - Q)":          "X - ZZ - Q",
}

print("=" * 60)
print("HARMONIC VALIDITY SCORES")
print("=" * 60)
print(f"{'Progression':<40} {'Score':>6}  {'Parse':>6}  {'Resolv':>7}  {'Tonic':>6}")
print("-" * 60)

for name, prog in test_cases.items():
    result = score_harmonic_validity(prog)
    print(f"{name:<40} {result['score']:>6.3f}  {result['parse_rate']:>6.3f}  {result['resolution']:>7.3f}  {result['tonic_anchor']:>6.3f}")


HARMONIC VALIDITY SCORES
Progression                               Score   Parse   Resolv   Tonic
------------------------------------------------------------
Sadness  (IV - V - I)                     0.750   1.000    0.500   0.500
Anger    (I - V - vi - IV - I)            0.700   1.000    0.000   1.000
Excitement (I - V - vi - IV - I)          0.700   1.000    0.000   1.000
Jazz     (Imaj7 - ii7 - V7 - I)           0.800   1.000    0.333   1.000
Minor    (i - iv - i)                     0.700   1.000    0.000   1.000
Invalid  (X - ZZ - Q)                     0.000   0.000    0.000   0.000


In [10]:
!pip install -q mir_eval

import mir_eval
import numpy as np

ROMAN_TO_HARTE = {
    'I':     'C:maj',    'II':    'D:maj',    'III':   'E:maj',
    'IV':    'F:maj',    'V':     'G:maj',    'VI':    'A:maj',
    'VII':   'B:maj',
    'i':     'C:min',    'ii':    'D:min',    'iii':   'E:min',
    'iv':    'F:min',    'v':     'G:min',    'vi':    'A:min',
    'vii':   'B:min',
    'I7':    'C:7',      'V7':    'G:7',      'IV7':   'F:7',
    'ii7':   'D:min7',   'vi7':   'A:min7',
    'Imaj7': 'C:maj7',   'IVmaj7':'F:maj7',
    'vii°':  'B:dim',    'ii°':   'D:dim',
}

def roman_to_harte(roman_str):
    return ROMAN_TO_HARTE.get(roman_str.strip(), 'N')

def parse_progression(progression_str):
    chords    = [c.strip() for c in progression_str.split(' - ')]
    harte     = [roman_to_harte(c) for c in chords]
    intervals = np.array([[i, i+1] for i in range(len(harte))], dtype=float)
    return intervals, harte

def score_chord_similarity(generated_progression, reference_progressions):
    """
    Uses mir_eval MIREX metric to compare generated chords
    against real human progressions from our database.
    Score: 0.0 to 1.0. Higher = more similar to real compositions.
    """
    try:
        gen_intervals, gen_labels = parse_progression(generated_progression)

        if all(l == 'N' for l in gen_labels):
            return {"score": 0.0, "comparisons": 0}

        scores = []
        for ref_prog in reference_progressions:
            try:
                ref_intervals, ref_labels = parse_progression(ref_prog)
                min_len = min(len(gen_labels), len(ref_labels))

                result = mir_eval.chord.evaluate(
                    ref_intervals[:min_len], ref_labels[:min_len],
                    gen_intervals[:min_len], gen_labels[:min_len]
                )
                # mirex is the standard competition metric for chord comparison
                scores.append(float(result['mirex']))

            except Exception as e:
                continue

        if not scores:
            return {"score": 0.0, "comparisons": 0}

        return {
            "score":       round(float(np.mean(scores)), 3),
            "comparisons": len(scores),
            "max_match":   round(float(np.max(scores)), 3),
            "min_match":   round(float(np.min(scores)), 3),
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"score": 0.0, "comparisons": 0}


def get_reference_progressions(emotion, db, top_n=10):
    _, results = query_db_for_emotion(emotion, db)
    return [r['roman'] for r in results[:top_n]]


# ── Test ─────────────────────────────────────────────────────────────
print("=" * 55)
print("CHORD SIMILARITY SCORES (mir_eval MIREX)")
print("=" * 55)

test_cases = [
    ("sadness", "IV - V - I"),
    ("joy",     "I - V - vi - IV - I"),
    ("anger",   "I - V - vi - IV - I"),
    ("sadness", "X - ZZ - Q"),
]

for emotion, progression in test_cases:
    refs   = get_reference_progressions(emotion, music_db)
    result = score_chord_similarity(progression, refs)
    print(f"\nEmotion    : {emotion.upper()}")
    print(f"Progression: {progression}")
    print(f"Similarity : {result['score']} (vs {result['comparisons']} references)")
    if result['comparisons'] > 0:
        print(f"Best match : {result['max_match']} | Worst: {result['min_match']}")


CHORD SIMILARITY SCORES (mir_eval MIREX)

Emotion    : SADNESS
Progression: IV - V - I
Similarity : 0.667 (vs 3 references)
Best match : 1.0 | Worst: 0.333

Emotion    : JOY
Progression: I - V - vi - IV - I
Similarity : 0.533 (vs 3 references)
Best match : 1.0 | Worst: 0.2

Emotion    : ANGER
Progression: I - V - vi - IV - I
Similarity : 0.756 (vs 3 references)
Best match : 1.0 | Worst: 0.6

Emotion    : SADNESS
Progression: X - ZZ - Q
Similarity : 0.0 (vs 0 references)


In [11]:
!pip install -q nltk

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from google import genai
from google.genai import types

def generate_reference_lyric(emotion, api_key):
    """
    Generates a plain emotion-only lyric with NO chord guidance.
    This is our automated reference text for METEOR comparison.
    """
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Write exactly 4 lines of lyrics expressing {emotion}. Output ONLY the 4 lines, nothing else.",
        config=types.GenerateContentConfig(
            temperature=0.75,
        )
    )
    return response.text.strip()


def score_meteor(generated_lyrics, reference_lyrics):
    """
    Compares MuseTune chord-guided lyrics against plain emotion-only lyrics.

    High score = both lyrics use similar rich emotional vocabulary
                 (chord guidance kept emotional consistency)
    Low score  = chord guidance pulled the lyrics in a different direction
                 (could mean chords added unique depth OR hurt emotion)

    Score: 0.0 to 1.0
    """
    try:
        generated_tokens = word_tokenize(generated_lyrics.lower())
        reference_tokens = word_tokenize(reference_lyrics.lower())

        score = meteor_score(
            [reference_tokens],
            generated_tokens
        )
        return round(score, 3)

    except Exception as e:
        print(f"METEOR error: {e}")
        return None


def interpret_meteor(score):
    if score is None:
        return "ERROR"
    elif score >= 0.5:
        return "Excellent"
    elif score >= 0.3:
        return "Good"
    elif score >= 0.15:
        return "Average"
    else:
        return "Poor"


def score_meteor_automated(emotion, chord_guided_lyrics, api_key):
    """
    Full automated METEOR pipeline:
    1. Generate reference lyric using emotion only (no chords)
    2. Compare chord-guided lyric against reference
    3. Return score
    """
    print(f"  Generating automated reference lyric for [{emotion.upper()}]...")
    reference = generate_reference_lyric(emotion, api_key)
    print(f"  Reference lyric:\n    {reference.replace(chr(10), chr(10)+'    ')}")

    score  = score_meteor(chord_guided_lyrics, reference)
    rating = interpret_meteor(score)
    return score, rating, reference


# ── Test on our generated lyrics ─────────────────────────────────────
test_cases = [
    ("sadness", """The memory of a lighter time
Still haunts me in this quiet space,
And leaves my spirit out of rhyme
With nowhere left to find a trace."""),

    ("anger", """You think I'm blind to every single lie?
The truth is staring, right there in your face,
My trust you shattered, watched it fall and die,
And now your consequence you will embrace."""),

    ("excitement", """The countdown starts, a steady sound,
My heart is pounding, fast and free,
The world we knew, now spinning round,
A brilliant future waits for me."""),
]

print("=" * 60)
print("METEOR SCORES (Fully Automated — No Human Reference)")
print("=" * 60)

for emotion, chord_lyrics in test_cases:
    print(f"\n--- {emotion.upper()} ---")
    print(f"Chord-guided lyrics:\n    {chord_lyrics.replace(chr(10), chr(10)+'    ')}")
    score, rating, _ = score_meteor_automated(emotion, chord_lyrics, MY_GEMINI_KEY)
    print(f"\nMETEOR Score : {score}")
    print(f"Rating       : {rating}")

print("\nNote: Compares chord-guided lyrics vs plain emotion-only lyrics.")
print("Both generated by AI — fully automated, no human reference needed.")


METEOR SCORES (Fully Automated — No Human Reference)

--- SADNESS ---
Chord-guided lyrics:
    The memory of a lighter time
    Still haunts me in this quiet space,
    And leaves my spirit out of rhyme
    With nowhere left to find a trace.
  Generating automated reference lyric for [SADNESS]...
  Reference lyric:
    The quiet rain mirrors the tears I can't shed,
    A heavy silence where your laughter once played.
    My heart's a hollow echo, filled with dread,
    Just a memory of joy, now faded and frayed.

METEOR Score : 0.181
Rating       : Average

--- ANGER ---
Chord-guided lyrics:
    You think I'm blind to every single lie?
    The truth is staring, right there in your face,
    My trust you shattered, watched it fall and die,
    And now your consequence you will embrace.
  Generating automated reference lyric for [ANGER]...
  Reference lyric:
    A burning rage consumes my soul,
    Your poisoned words have taken their toll.
    I see the lies behind your smile,
    And w

In [12]:
!pip install -q transformers torch

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import math

print("Loading GPT-2 for perplexity scoring...")

gpt2_model    = GPT2LMHeadModel.from_pretrained('gpt2')
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model.eval()

print("GPT-2 loaded.\n")


def score_perplexity(text):
    """
    Measures how natural/fluent a piece of text is using GPT-2.

    Perplexity interpretation:
    - Under  50  → Very fluent, natural writing
    - 50–100     → Good, readable writing
    - 100–200    → Slightly awkward or unusual phrasing
    - Above 200  → Unnatural, broken text
    """
    try:
        encodings = gpt2_tokenizer(text, return_tensors='pt')
        input_ids = encodings.input_ids

        with torch.no_grad():
            outputs = gpt2_model(input_ids, labels=input_ids)
            loss    = outputs.loss

        perplexity = math.exp(loss.item())
        return round(perplexity, 2)

    except Exception as e:
        print(f"Error scoring text: {e}")
        return None


def interpret_perplexity(score):
    """Returns a human-readable label for a perplexity score."""
    if score is None:
        return "ERROR"
    elif score < 50:
        return "Excellent"
    elif score < 100:
        return "Good"
    elif score < 200:
        return "Average"
    else:
        return "Poor"


# ── Test on our generated lyrics ─────────────────────────────────────
test_lyrics = {
    "Sadness lyrics": """The memory of a lighter time
Still haunts me in this quiet space,
And leaves my spirit out of rhyme
With nowhere left to find a trace.""",

    "Anger lyrics": """You think I'm blind to every single lie?
The truth is staring, right there in your face,
My trust you shattered, watched it fall and die,
And now your consequence you will embrace.""",

    "Excitement lyrics": """The countdown starts, a steady sound,
My heart is pounding, fast and free,
The world we knew, now spinning round,
A brilliant future waits for me.""",

    "Random broken text": "xyz potato 1234 the sky is green banana",

    "Normal English sentence": "I went to the store to buy some milk and bread.",
}

print("=" * 60)
print("LYRIC FLUENCY SCORES (Perplexity via GPT-2)")
print("=" * 60)
print(f"{'Sample':<25} {'Perplexity':>12}  {'Rating':>10}")
print("-" * 60)

for name, text in test_lyrics.items():
    score  = score_perplexity(text)
    rating = interpret_perplexity(score)
    print(f"{name:<25} {score:>12.2f}  {rating:>10}")

print("\nNote: Lower perplexity = more natural and fluent text.")


Loading GPT-2 for perplexity scoring...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GPT-2 loaded.

LYRIC FLUENCY SCORES (Perplexity via GPT-2)
Sample                      Perplexity      Rating
------------------------------------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Sadness lyrics                  100.90     Average
Anger lyrics                     84.82        Good
Excitement lyrics                81.01        Good
Random broken text             1424.40        Poor
Normal English sentence          17.90   Excellent

Note: Lower perplexity = more natural and fluent text.


In [13]:
def score_emotional_alignment(target_emotion, generated_lyrics):
    """
    Uses RoBERTa to check if the generated lyrics actually express
    the intended target emotion.

    Process:
    1. Run generated lyrics through RoBERTa
    2. Check if detected emotion matches target emotion
    3. Return alignment score + confidence

    Score: 0.0 to 1.0
    - 1.0 = perfect match (lyrics express exact target emotion)
    - 0.5 = partial match (lyrics express a related emotion)
    - 0.0 = mismatch (lyrics express a completely different emotion)
    """
    try:
        # Run lyrics through RoBERTa
        results        = emotion_classifier(generated_lyrics)
        top_emotion    = results[0][0]['label']
        top_confidence = results[0][0]['score']
        all_emotions   = {r['label']: r['score'] for r in results[0]}

        # Check for exact match
        if top_emotion == target_emotion:
            alignment_score = round(top_confidence, 3)
            match_type      = "EXACT MATCH"

        # Check for related emotion match
        # (e.g. target=sadness, detected=grief → still related)
        elif GOEMOTIONS_TO_MUSICAL.get(top_emotion) == GOEMOTIONS_TO_MUSICAL.get(target_emotion):
            alignment_score = round(top_confidence * 0.75, 3)
            match_type      = "RELATED MATCH"

        # Check if target emotion appears in top 3 even if not #1
        elif target_emotion in all_emotions:
            alignment_score = round(all_emotions[target_emotion] * 0.5, 3)
            match_type      = "WEAK MATCH"

        else:
            alignment_score = 0.0
            match_type      = "MISMATCH"

        return {
            "score":           alignment_score,
            "match_type":      match_type,
            "target_emotion":  target_emotion,
            "detected_emotion": top_emotion,
            "confidence":      round(top_confidence, 3),
        }

    except Exception as e:
        print(f"Alignment error: {e}")
        return {"score": 0.0, "match_type": "ERROR"}


# ── Test it ───────────────────────────────────────────────────────────
print("=" * 60)
print("EMOTIONAL ALIGNMENT SCORES (RoBERTa)")
print("=" * 60)

test_cases = [
    ("sadness", """The memory of a lighter time
Still haunts me in this quiet space,
And leaves my spirit out of rhyme
With nowhere left to find a trace."""),

    ("anger", """You think I'm blind to every single lie?
The truth is staring, right there in your face,
My trust you shattered, watched it fall and die,
And now your consequence you will embrace."""),

    ("excitement", """The countdown starts, a steady sound,
My heart is pounding, fast and free,
The world we knew, now spinning round,
A brilliant future waits for me."""),

    # Mismatch test — happy lyrics for sad target
    ("sadness", """Everything is sunshine, birds are singing loud,
Dancing in the rain beneath a silver cloud,
Life is full of laughter, joy is all around,
Happiness and love are all that I have found."""),
]

for target, lyrics in test_cases:
    result = score_emotional_alignment(target, lyrics)
    print(f"\nTarget emotion  : {result['target_emotion'].upper()}")
    print(f"Detected emotion: {result['detected_emotion'].upper()}")
    print(f"Match type      : {result['match_type']}")
    print(f"Alignment score : {result['score']}")


EMOTIONAL ALIGNMENT SCORES (RoBERTa)

Target emotion  : SADNESS
Detected emotion: NEUTRAL
Match type      : WEAK MATCH
Alignment score : 0.082

Target emotion  : ANGER
Detected emotion: NEUTRAL
Match type      : MISMATCH
Alignment score : 0.0

Target emotion  : EXCITEMENT
Detected emotion: EXCITEMENT
Match type      : EXACT MATCH
Alignment score : 0.417

Target emotion  : SADNESS
Detected emotion: JOY
Match type      : MISMATCH
Alignment score : 0.0


In [14]:
!pip install -q gradio

import gradio as gr

MAJOR_SCALES = {
    'C':  ['C',  'D',  'E',  'F',  'G',  'A',  'B'],
    'C#': ['C#', 'D#', 'F',  'F#', 'G#', 'A#', 'C'],
    'Db': ['Db', 'Eb', 'F',  'Gb', 'Ab', 'Bb', 'C'],
    'D':  ['D',  'E',  'F#', 'G',  'A',  'B',  'C#'],
    'Eb': ['Eb', 'F',  'G',  'Ab', 'Bb', 'C',  'D'],
    'E':  ['E',  'F#', 'G#', 'A',  'B',  'C#', 'D#'],
    'F':  ['F',  'G',  'A',  'Bb', 'C',  'D',  'E'],
    'F#': ['F#', 'G#', 'A#', 'B',  'C#', 'D#', 'F'],
    'G':  ['G',  'A',  'B',  'C',  'D',  'E',  'F#'],
    'Ab': ['Ab', 'Bb', 'C',  'Db', 'Eb', 'F',  'G'],
    'A':  ['A',  'B',  'C#', 'D',  'E',  'F#', 'G#'],
    'Bb': ['Bb', 'C',  'D',  'Eb', 'F',  'G',  'A'],
    'B':  ['B',  'C#', 'D#', 'E',  'F#', 'G#', 'A#'],
}

def roman_to_chord(roman_str, key_root='C'):
    try:
        roman_str = roman_str.strip()
        is_minor  = roman_str[0].islower()
        upper     = roman_str.upper()
        base      = None
        suffix    = ''
        for r in ['VII', 'VI', 'IV', 'III', 'II', 'V', 'I']:
            if upper.startswith(r):
                base   = r
                suffix = roman_str[len(r):]
                break
        if not base:
            return roman_str
        INDEX     = {'I':0, 'II':1, 'III':2, 'IV':3, 'V':4, 'VI':5, 'VII':6}
        scale     = MAJOR_SCALES.get(key_root, MAJOR_SCALES['C'])
        root_note = scale[INDEX[base]]
        suffix_up = suffix.upper()
        if '°' in suffix or 'DIM' in suffix_up:
            quality = 'dim'
        elif is_minor:
            quality = 'm'
        else:
            quality = ''
        if 'MAJ7' in suffix_up:
            quality += 'maj7'
        elif '7' in suffix:
            quality += '7'
        return root_note + quality
    except:
        return roman_str


def progression_to_real_chords(progression_str, key_root='C'):
    chords = [c.strip() for c in progression_str.split(' - ')]
    return '  -  '.join(roman_to_chord(c, key_root) for c in chords)


# ── Main app function ─────────────────────────────────────────────────
def musetune_app(journal_entry, selected_key):
    if not journal_entry or journal_entry.strip() == "":
        return "Please enter some text.", "", "", "", "", "", "", "", ""

    try:
        # Step 1: Detect emotion
        results     = emotion_classifier(journal_entry)
        top_emotion = results[0][0]['label']
        confidence  = results[0][0]['score']
        emotion_str = f"{top_emotion.upper()}   (confidence: {confidence:.0%})"

        # Step 2: Query database
        musical_emotion, db_results = query_db_for_emotion(top_emotion, music_db)
        best_chords = db_results[0]['roman'] if db_results else "I - V - vi - IV"

        # Convert to real chord names
        real_chords = progression_to_real_chords(best_chords, selected_key)
        chords_str  = f"{real_chords}\n\n(Roman numerals: {best_chords})"

        # Example songs
        example_songs = ""
        if db_results:
            for ex in db_results[0]['examples'].itertuples():
                example_songs += f"• {ex.artist.title()} — '{ex.song.title()}'\n"

        # Step 3: Generate lyrics
        lyrics = generate_lyrics(top_emotion, best_chords, MY_GEMINI_KEY)

        # ── Score 1: Harmonic Validity (Music21) ──────────────────────
        h_score      = score_harmonic_validity(best_chords)
        harmonic_str = (
            f"Overall: {h_score['score']}  |  "
            f"Parse: {h_score['parse_rate']}  |  "
            f"Resolution: {h_score['resolution']}  |  "
            f"Tonic: {h_score['tonic_anchor']}"
        )

        # ── Score 2: Chord Similarity (mir_eval) ──────────────────────
        refs       = get_reference_progressions(top_emotion, music_db)
        sim_result = score_chord_similarity(best_chords, refs)
        sim_str    = (
            f"MIREX Score: {sim_result['score']}  |  "
            f"References: {sim_result['comparisons']}  |  "
            f"Best match: {sim_result.get('max_match', 'N/A')}"
        )

        # ── Score 3: Lyric Fluency (GPT-2 Perplexity) ─────────────────
        p_score     = score_perplexity(lyrics)
        fluency_str = (
            f"Perplexity: {p_score}  |  "
            f"Rating: {interpret_perplexity(p_score)}"
        )

        # ── Score 4: METEOR Score (NLTK) ──────────────────────────────
        meteor, m_rating, _ = score_meteor_automated(top_emotion, lyrics, MY_GEMINI_KEY)
        meteor_str  = (
            f"METEOR: {meteor}  |  "
            f"Rating: {m_rating}"
        )

        # ── Score 5: Emotional Alignment (RoBERTa) ────────────────────
        align_result  = score_emotional_alignment(top_emotion, lyrics)
        alignment_str = (
            f"Score: {align_result['score']}  |  "
            f"Match: {align_result['match_type']}  |  "
            f"Detected: {align_result['detected_emotion'].upper()}"
        )

        return (
            emotion_str,
            chords_str,
            example_songs.strip(),
            lyrics,
            harmonic_str,
            sim_str,
            fluency_str,
            meteor_str,
            alignment_str
        )

    except Exception as e:
        return f"Error: {e}", "", "", "", "", "", "", "", ""


# ── Gradio UI ─────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="MuseTune") as app:

    gr.Markdown("""
    # 🎵 MuseTune
    ### Turn your feelings into a song
    *Powered by RoBERTa · HookTheory · Gemini · Music21 · mir_eval · NLTK*
    """)

    with gr.Row():
        with gr.Column(scale=2):
            journal_input = gr.Textbox(
                label="How are you feeling today?",
                placeholder="Write anything — a journal entry, a thought, a feeling...",
                lines=4
            )
            key_selector = gr.Dropdown(
                label="Select a Musical Key",
                choices=['C','C#','Db','D','Eb','E','F','F#','G','Ab','A','Bb','B'],
                value='C',
                info="Beginners: start with C. Each key gives different chord names."
            )
            submit_btn = gr.Button(
                "✨ Generate My Song",
                variant="primary",
                size="lg"
            )

        with gr.Column(scale=1):
            emotion_output = gr.Textbox(
                label="🎭 Detected Emotion",
                interactive=False
            )
            chords_output = gr.Textbox(
                label="🎸 Chord Progression",
                interactive=False,
                lines=3
            )
            songs_output = gr.Textbox(
                label="🎵 Real Songs Using These Chords",
                interactive=False,
                lines=3
            )

    gr.Markdown("### 🎼 Generated Lyrics")
    lyrics_output = gr.Textbox(
        label="",
        interactive=False,
        lines=6
    )

    gr.Markdown("### 📊 Evaluation Scores")

    with gr.Row():
        harmonic_output = gr.Textbox(
            label="🎹 Harmonic Validity (Music21)",
            interactive=False
        )
        similarity_output = gr.Textbox(
            label="🎯 Chord Similarity (mir_eval MIREX)",
            interactive=False
        )

    with gr.Row():
        fluency_output = gr.Textbox(
            label="📝 Lyric Fluency (GPT-2 Perplexity)",
            interactive=False
        )
        meteor_output = gr.Textbox(
            label="💬 Lyric Quality (METEOR Score)",
            interactive=False
        )

    with gr.Row():
        alignment_output = gr.Textbox(
            label="🎭 Emotional Alignment (RoBERTa)",
            interactive=False
        )

    gr.Markdown("""
    ---
    **How MuseTune evaluates itself — 5 fully automated metrics:**

    | Metric | Tool | What it checks |
    | --- | --- | --- |
    | Harmonic Validity | Music21 | Are the chords theoretically correct? |
    | Chord Similarity | mir_eval MIREX | Do chords match real human compositions? |
    | Lyric Fluency | GPT-2 Perplexity | Are the lyrics natural and well written? |
    | Lyric Quality | METEOR | Are the lyrics vocabulary rich? |
    | Emotional Alignment | RoBERTa | Do the lyrics express the right emotion? |
    """)

    submit_btn.click(
        fn=musetune_app,
        inputs=[journal_input, key_selector],
        outputs=[
            emotion_output,
            chords_output,
            songs_output,
            lyrics_output,
            harmonic_output,
            similarity_output,
            fluency_output,
            meteor_output,
            alignment_output
        ]
    )

print("Launching MuseTune app...")
app.launch(share=True, debug=False)


/tmp/ipykernel_13543/3115631472.py:145: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="MuseTune") as app:


Launching MuseTune app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9687fea4672f71bbdb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
